# 14b · Importare un aggiornamento in Azure Update Service

Un aggiornamento software sono **due cose**: i **file** che il device installa e l'**import manifest**, un JSON che li descrive: identità (provider, nome, versione), **compatibilità**, istruzioni di installazione e, per ogni file, **dimensione e hash**.
Qui prepariamo **due versioni** di un piccolo script di termostato (quello del notebook 16, ma questa volta lo consegna un job):
- **2.0.0**: funziona (nuovo setpoint e allarme);
- **3.0.0**: **difettosa** (lancia un errore): serve a vedere cosa fa il device quando l'installazione non riesce.

Comandi: `az iot adr ns su software-update init v5` crea il manifest (calcola da solo dimensione e hash), `stage --then-import` carica file e manifest nello storage del notebook 14a e li importa. Importando, **Azure Update Service firma il manifest**: il device verificherà quella firma.
La **compatibilità** dice a chi può andare: `manufacturer=Contoso`, `model=RobotV1`, gli stessi valori che il device ha in ADR.

**Serve:** il notebook 14a eseguito (Update Instance collegata, storage, ruolo).
**Da dire.** "Il manifest è la carta d'identità dell'aggiornamento: chi è, a chi va, e l'impronta dei file. La firma la mette il servizio, non io."


In [ ]:
# Carica la libreria e legge lo scenario (notebooks\_scenario.psd1): da qui in poi si usa $cfg
. ([scriptblock]::Create([IO.File]::ReadAllText((Get-ChildItem . , .\notebooks , .. -Filter _start.ps1 -ErrorAction SilentlyContinue | Select-Object -First 1).FullName)))

## 1 · I file dell'aggiornamento
Due script. Si scrivono in `artifacts\updates` (non finiscono nel repository).


In [ ]:
$provider = 'Contoso'; $nome = 'termostato'    # l'identita' dell'aggiornamento: provider / nome / versione
$ns = $adrName
$dir = Join-Path $out 'updates\termostato'
New-Item -ItemType Directory -Force $dir | Out-Null
$utf8 = [Text.UTF8Encoding]::new($false)
$v2 = @'
# thermostat 2.0.0
function Get-Telemetry {
    $setpoint = 19.0; $temperature = 26.1
    [pscustomobject]@{ payloadVersion = '2.0.0'; setpoint = $setpoint; temperature = $temperature; alarm = ($temperature -gt 25) }
}
'@
$v3 = @'
# thermostat 3.0.0 (difettoso)
function Get-Telemetry { throw 'sensore non inizializzato' }
'@
[IO.File]::WriteAllText((Join-Path $dir 'thermostat-2.0.0.ps1'), $v2, $utf8)
[IO.File]::WriteAllText((Join-Path $dir 'thermostat-3.0.0.ps1'), $v3, $utf8)
Get-ChildItem $dir | Select-Object Name, Length | Format-Table | Out-String


## 2 · L'import manifest
`init v5` legge i file e scrive il manifest con **dimensione e hash SHA-256**. La compatibilità e il passo di installazione (`microsoft/script:1`) si dichiarano qui.


In [ ]:
foreach ($v in '2.0.0', '3.0.0') {
    $manifest = az iot adr ns su software-update init v5 `
        --update-provider $provider `
        --update-name $nome `
        --update-version $v `
        --compat manufacturer=Contoso model=RobotV1 `
        --step handler=microsoft/script:1 `
        --file "path=$(Join-Path $dir "thermostat-$v.ps1")" `
        --description "Termostato $v" `
        --only-show-errors
    Assert-Az "Creazione del manifest $v"
    [IO.File]::WriteAllText((Join-Path $dir "$nome.$v.importmanifest.json"), ($manifest -join "`n"), $utf8)
}
Get-Content (Join-Path $dir "$nome.2.0.0.importmanifest.json") -Raw


## 3 · Carica e importa
`stage` carica manifest e file nel container `updates` dello storage e, con `--then-import`, li importa con un indirizzo di sola lettura a tempo. Un aggiornamento già importato non si reimporta: la cella lo salta.


In [ ]:
$importati = @(az iot adr ns su software-update list --ns $ns --resource-group $rg --query "[].updateId.version" -o tsv --only-show-errors)
foreach ($v in '2.0.0', '3.0.0') {
    if ($v -in $importati) { "$nome $v : gia' importato"; continue }
    az iot adr ns su software-update stage `
        --ns $ns `
        --resource-group $rg `
        --manifest-path (Join-Path $dir "$nome.$v.importmanifest.json") `
        --storage-account $update.Storage `
        --storage-container $update.Container `
        --then-import true `
        --only-show-errors
    Assert-Az "Import di $nome $v"
    "$nome $v : importato"
}


## 4 · Cosa c'è nel servizio
La lista degli aggiornamenti importati e il blob nello storage. `scanResult: OptedOut` perché non abbiamo chiesto la scansione antimalware (`--enable-scan`).


In [ ]:
az iot adr ns su software-update list --ns $ns --resource-group $rg --query "[].{provider:updateId.provider, nome:updateId.name, versione:updateId.version, compatibilita:compatibility[0], importato:importedDateTime}" -o table --only-show-errors
az storage blob list --account-name $update.Storage --container-name $update.Container --auth-mode login --query "[].name" -o tsv --only-show-errors


## Cosa abbiamo visto
- Un aggiornamento è **file + import manifest**; il manifest lo crea la CLI (non il portale) con dimensione e hash.
- Il manifest dice **a chi può andare** (`compatibility`: produttore e modello) e **come installare** (`handler`).
- L'import lo **firma** Azure Update Service. Un aggiornamento con la stessa identità non si reimporta (`UpdateAlreadyExists`): per cambiare un file serve una **nuova versione**.

**Prossimo notebook: 14c, un job che consegna l'aggiornamento al device.**
